In [1]:
import pandas as pd

In [2]:
df = pd.read_excel('wibor_all.xlsx')
df['currency'] = 'PLN'

In [3]:
df = df.melt(id_vars=['date', 'currency'], value_vars=['WIB1D', 'WIB1M', 'WIB3M', 'WIB6M', 'WIB1Y'], var_name='ir_type', value_name='rate')

In [4]:
df['tenor'] = df['ir_type'].str.replace('WIB', '')

In [5]:
# ── IRS PLN swap curve (2Y..10Y) — only available bi-monthly (mid-month +
# month-end), so interpolate to daily before merging with the WIBOR panel.
# WIBOR's own 1Y stays authoritative for the 1Y point (money-market deposit
# convention, matches the front end); IRS's 1Y column is dropped to avoid a
# duplicate/conflicting tenor label, so only the genuinely new tenors
# (2Y,3Y,4Y,5Y,7Y,10Y) are added.
irs = pd.read_excel('pln_irs_all.xlsx').rename(columns={'Data': 'date'})
irs_tenor_cols = [c for c in irs.columns if c != 'date' and c != 'IRS PLN 1Y']

irs = irs.set_index('date').sort_index()
irs = irs[irs_tenor_cols] * 100.0  # decimal fraction -> percentage, matches WIBOR's convention

# Reindex onto WIBOR's own business-day calendar (not every calendar day) --
# otherwise weekend/holiday dates get IRS rows but no WIBOR short end, and
# curve_generation_job crashes on the missing 1D/1M/../1Y lookup for those.
wibor_bdays = pd.Index(sorted(df['date'].unique()))
irs_daily = irs.reindex(wibor_bdays).interpolate(method='linear').rename_axis('date').reset_index()

irs_long = irs_daily.melt(id_vars=['date'], value_vars=irs_tenor_cols, var_name='ir_type', value_name='rate')
irs_long['tenor'] = irs_long['ir_type'].str.replace('IRS PLN ', '')
irs_long['currency'] = 'PLN'
irs_long = irs_long.dropna(subset=['rate'])
irs_long = irs_long[['date', 'currency', 'ir_type', 'rate', 'tenor']]

In [6]:
df = pd.concat([df, irs_long], ignore_index=True).sort_values(['date', 'tenor']).reset_index(drop=True)

In [7]:
df.to_excel('curve_input.xlsx', index=False)

In [8]:
df[df['date'] == '2024-06-03']['date']

29791   2024-06-03
29792   2024-06-03
29793   2024-06-03
29794   2024-06-03
29795   2024-06-03
29796   2024-06-03
29797   2024-06-03
29798   2024-06-03
29799   2024-06-03
29800   2024-06-03
29801   2024-06-03
Name: date, dtype: datetime64[us]

In [9]:
df.isna().sum()

date        0
currency    0
ir_type     0
rate        0
tenor       0
dtype: int64